# 🔧 Feature Engineering - Amaliy Mashg'ulot

**Maqsad:** Real dataset'larda feature engineering texnikalarini amalda qo'llash

**Exercises:**
1. Feature Selection on High-Dimensional Data
2. Scaling Impact on Different Algorithms
3. Feature Engineering Pipeline
4. Complete ML Project with FE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, fetch_california_housing, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif, RFE, SelectFromModel, VarianceThreshold
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")

---

## Exercise 1: Feature Selection on High-Dimensional Data

**Dataset:** Breast Cancer (30 features)

**Tasks:**
1. Load dataset va EDA
2. 3 ta selection usulini qo'llang (Filter, Wrapper, Embedded)
3. Har bir usul bilan model train qiling
4. Natijalarni taqqoslang
5. Eng yaxshi usulni tanlang

In [ ]:
# Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(f"Dataset shape: {X.shape}")
print(f"Target distribution: {np.bincount(y)}")
X.head()

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO: 1. Filter Method - SelectKBest (k=10)
selector_filter = SelectKBest(score_func=f_classif, k=10)
X_train_filter = selector_filter.fit_transform(X_train, y_train)
X_test_filter = selector_filter.transform(X_test)

# Train model
model_filter = LogisticRegression(max_iter=10000, random_state=42)
model_filter.fit(X_train_filter, y_train)
y_pred_filter = model_filter.predict(X_test_filter)
acc_filter = accuracy_score(y_test, y_pred_filter)

print(f"Filter Method Accuracy: {acc_filter:.4f}")
print(f"Selected features: {X.columns[selector_filter.get_support()].tolist()}")

In [ ]:
# TODO: 2. Wrapper Method - RFE (n=10)
model_rfe = LogisticRegression(max_iter=10000, random_state=42)
selector_rfe = RFE(estimator=model_rfe, n_features_to_select=10)
X_train_rfe = selector_rfe.fit_transform(X_train, y_train)
X_test_rfe = selector_rfe.transform(X_test)

# Train model
model_rfe_final = LogisticRegression(max_iter=10000, random_state=42)
model_rfe_final.fit(X_train_rfe, y_train)
y_pred_rfe = model_rfe_final.predict(X_test_rfe)
acc_rfe = accuracy_score(y_test, y_pred_rfe)

print(f"RFE Method Accuracy: {acc_rfe:.4f}")
print(f"Selected features: {X.columns[selector_rfe.support_].tolist()}")

In [ ]:
# TODO: 3. Embedded Method - Random Forest Feature Importance
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)

selector_embedded = SelectFromModel(model_rf, threshold='median', prefit=True)
X_train_embedded = selector_embedded.transform(X_train)
X_test_embedded = selector_embedded.transform(X_test)

# Train model
model_embedded = LogisticRegression(max_iter=10000, random_state=42)
model_embedded.fit(X_train_embedded, y_train)
y_pred_embedded = model_embedded.predict(X_test_embedded)
acc_embedded = accuracy_score(y_test, y_pred_embedded)

print(f"Embedded Method Accuracy: {acc_embedded:.4f}")
print(f"Selected features: {X.columns[selector_embedded.get_support()].tolist()}")

In [ ]:
# Compare results
results = pd.DataFrame({
    'Method': ['Filter (SelectKBest)', 'Wrapper (RFE)', 'Embedded (RF Importance)'],
    'Accuracy': [acc_filter, acc_rfe, acc_embedded],
    'Features Selected': [10, 10, X_train_embedded.shape[1]]
})

print("\n" + "="*70)
print("FEATURE SELECTION METHODS COMPARISON")
print("="*70)
print(results.to_string(index=False))

# Visualization
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.bar(results['Method'], results['Accuracy'], 
       color=['skyblue', 'coral', 'lightgreen'], 
       edgecolor='black', alpha=0.7)
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Feature Selection Methods Performance', fontsize=14, fontweight='bold')
ax.set_ylim(0.9, 1.0)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

---

## Exercise 2: Scaling Impact Analysis

**Tasks:**
1. California Housing dataset yuklang
2. 4 ta scaler test qiling (None, Standard, MinMax, Robust)
3. 3 ta model bilan test qiling (LinearReg, KNN, DecisionTree)
4. Heatmap yarating (model vs scaler)
5. Insightlar chiqaring

In [ ]:
# Load dataset
housing = fetch_california_housing()
X_house = pd.DataFrame(housing.data, columns=housing.feature_names)
y_house = housing.target

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

print(f"Dataset shape: {X_house.shape}")
print(f"Target range: [{y_house.min():.2f}, {y_house.max():.2f}]")

In [ ]:
# TODO: Test different scalers with different models
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

scalers = {
    'No Scaling': None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

models = {
    'Linear Regression': LinearRegression(),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42)
}

results_scaling = []

for model_name, model in models.items():
    for scaler_name, scaler in scalers.items():
        # Apply scaling
        if scaler is None:
            X_train_scaled = X_train_h
            X_test_scaled = X_test_h
        else:
            X_train_scaled = scaler.fit_transform(X_train_h)
            X_test_scaled = scaler.transform(X_test_h)
        
        # Train and evaluate
        model.fit(X_train_scaled, y_train_h)
        y_pred = model.predict(X_test_scaled)
        r2 = r2_score(y_test_h, y_pred)
        
        results_scaling.append({
            'Model': model_name,
            'Scaler': scaler_name,
            'R²': r2
        })

df_scaling = pd.DataFrame(results_scaling)
print(df_scaling)

In [ ]:
# Visualization: Heatmap
pivot = df_scaling.pivot(index='Model', columns='Scaler', values='R²')

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', 
            cbar_kws={'label': 'R² Score'}, ax=ax, linewidths=1, linecolor='black')
ax.set_title('Scaling Impact on Regression Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✅ Key Insights:")
print("   - KNN: Scaling CRITICAL! (distance-based)")
print("   - Linear Regression: Scaling helps slightly")
print("   - Decision Tree: Scaling NOT needed! (split-based)")

---

## Exercise 3: Complete Feature Engineering Pipeline

**Tasks:**
1. Mixed dataset yarating (numerical + categorical)
2. Complete pipeline building:
   - Numerical: Impute → Scale
   - Categorical: Impute → One-Hot
   - Feature Selection
   - Model
3. Cross-validation
4. Compare with baseline (no pipeline)

In [ ]:
# Create mixed dataset
np.random.seed(42)
n = 1000

data_mixed = pd.DataFrame({
    'Age': np.random.randint(18, 70, n),
    'Income': np.random.randint(20000, 150000, n),
    'Credit_Score': np.random.randint(300, 850, n),
    'Education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n),
    'City': np.random.choice(['NY', 'LA', 'SF', 'Chicago'], n),
    'Target': np.random.choice([0, 1], n, p=[0.4, 0.6])
})

# Add missing values
data_mixed.loc[np.random.choice(data_mixed.index, 50), 'Income'] = np.nan
data_mixed.loc[np.random.choice(data_mixed.index, 30), 'Education'] = np.nan

print(f"Dataset shape: {data_mixed.shape}")
print(f"Missing values:\n{data_mixed.isnull().sum()}")
data_mixed.head()

In [ ]:
# TODO: Build complete pipeline
X_mixed = data_mixed.drop('Target', axis=1)
y_mixed = data_mixed['Target']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_mixed, y_mixed, test_size=0.2, random_state=42
)

# Define columns
numerical_features = ['Age', 'Income', 'Credit_Score']
categorical_features = ['Education', 'City']

# Numerical pipeline
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Complete pipeline
pipeline_complete = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest(f_classif, k=8)),
    ('classifier', LogisticRegression(max_iter=10000, random_state=42))
])

# Train pipeline
pipeline_complete.fit(X_train_m, y_train_m)
y_pred_pipe = pipeline_complete.predict(X_test_m)
acc_pipe = accuracy_score(y_test_m, y_pred_pipe)

print(f"\n✅ Pipeline Accuracy: {acc_pipe:.4f}")

# Cross-validation
cv_scores = cross_val_score(pipeline_complete, X_mixed, y_mixed, cv=5, scoring='accuracy')
print(f"   CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

---

## Exercise 4: Complete ML Project with Feature Engineering

**Dataset:** Titanic-style survival prediction

**Tasks:**
1. Create realistic dataset
2. Apply ALL feature engineering techniques:
   - Feature creation (Age groups, Income brackets)
   - Feature encoding (multiple methods)
   - Feature selection
   - Feature scaling
3. Compare with baseline
4. Interpretability analysis

In [ ]:
# TODO: Complete this exercise on your own!
# Create dataset, apply FE, train models, compare results

print("\n💡 HINTS:")
print("1. Create features: Age_Group, Income_Bracket, FamilySize")
print("2. Encode: Education (Label), City (One-Hot), Job (Target)")
print("3. Select: Use multiple methods and compare")
print("4. Scale: Test impact on KNN, SVM, LR")
print("5. Compare: Baseline vs Full FE pipeline")
print("\nGood luck! 🚀")

---

## Summary

**Bizdagi amaliy mashg'ulotda o'rgandik:**

✅ **Feature Selection:**
- Filter, Wrapper, Embedded usullarini taqqoslash
- Qaysi usul qachon yaxshi ishlashini ko'rish

✅ **Scaling Impact:**
- Distance-based modellar uchun critical
- Tree-based modellar uchun kerak emas

✅ **Complete Pipeline:**
- Numerical va categorical features alohida handle qilish
- Data leakage'dan qochish
- Reproducibility va deployment ready

✅ **Best Practices:**
- Always use pipelines
- Validate with cross-validation
- Compare with baseline
- Document your transformations

---

**🚀 Keyingi qadamlar:**
1. `homework.md` - Mustaqil vazifalar
2. `feature_engineering_guide.md` - Quick reference
3. Real projects'larda qo'llang!